# 投资组合回测框架使用示例

本 notebook 演示如何使用可扩展的投资组合回测框架

In [1]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np

from portfolio_backtest import (
    BacktestEngine,
    RiskParityStrategy,
    MeanVarianceStrategy
)
from portfolio_backtest.visualization import BacktestVisualizer
from portfolio_backtest.utils import load_price_data

## 1. 加载数据

In [2]:
# # 加载价格数据
# price_df = load_price_data('./market_close.csv')
# print(f"数据形状: {price_df.shape}")
# print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
# print(f"\n资产列表:")
# for col in price_df.columns:
#     print(f"  - {col}")

# price_df.head()

In [3]:
# 加载价格数据
price_df = pd.read_excel('./market_close.xlsx')
price_df.columns = price_df.iloc[2]
price_df = price_df.iloc[4:]
price_df['日期'] = pd.to_datetime(price_df['日期'])
price_df = price_df.set_index('日期')
print(f"数据形状: {price_df.shape}")
print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
print(f"\n资产列表:")
for col in price_df.columns:
    print(f"  - {col}")

price_df.head()

数据形状: (2550, 10)
日期范围: 2015-09-01 00:00:00 到 2026-03-06 00:00:00

资产列表:
  - 上证指数
  - 创业板指
  - 纳斯达克指数
  - 道琼斯工业平均
  - 中证转债
  - 中债-商业银行二级资本债券财富(总值)指数
  - 中债-新综合财富(1年以下)指数
  - 中债-新综合财富(1-3年)指数
  - SGE黄金9999
  - ICE布油


2,上证指数,创业板指,纳斯达克指数,道琼斯工业平均,中证转债,中债-商业银行二级资本债券财富(总值)指数,中债-新综合财富(1年以下)指数,中债-新综合财富(1-3年)指数,SGE黄金9999,ICE布油
日期,,,,,,,,,,
2015-09-01,3166.6239,1889.491,29556.128472,102375.19292,291.3647,108.0298,150.7559,161.255,234.6,320.098792
2015-09-02,3160.167,1855.032,30218.897762,104025.844422,289.3253,108.1052,150.7703,161.2732,234.9,331.900323
2015-09-07,3080.4201,1893.521,29782.236928,102385.372992,292.4632,108.0971,150.8428,161.3674,231,314.232128
2015-09-08,3170.4522,2001.156,30622.641327,104957.766252,303.3656,108.0641,150.8641,161.3759,230.58,324.813456
2015-09-09,3243.0889,2071.717,30266.751696,103424.716624,310.0523,107.8927,150.8783,161.3896,231.1,313.260336


## 2. 风险平价策略回测

In [29]:
# 创建风险平价策略
rp_strategy = RiskParityStrategy(
    lookback=120,           # 12日回看窗口
    rebalance_freq='QE'    # ME月末调仓, QE 季末调仓
)

# 创建回测引擎
engine = BacktestEngine(
    init_cash=1_000_000,
    freq='1D'
)

# 运行回测
rp_result = engine.run(rp_strategy, price_df)

d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [34]:
rp_result.weights

2,上证指数,创业板指,纳斯达克指数,道琼斯工业平均,中证转债,中债-商业银行二级资本债券财富(总值)指数,中债-新综合财富(1年以下)指数,中债-新综合财富(1-3年)指数,SGE黄金9999,ICE布油
2016-03-31,0.005002,0.003131,0.006877,0.008234,0.007869,8.335266e-02,6.312387e-01,2.357210e-01,1.390626e-02,4.667467e-03
2016-06-30,0.004601,0.003022,0.007803,0.010537,0.008531,8.321786e-02,6.472921e-01,2.134481e-01,1.707774e-02,4.469455e-03
2016-09-30,0.136658,0.089515,0.172441,0.199901,0.330829,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.065511e-02
2016-12-30,0.019974,0.014695,0.023790,0.029090,0.027856,8.627270e-02,5.760123e-01,2.125812e-01,0.000000e+00,9.728481e-03
2017-03-31,0.025391,0.017032,0.027510,0.033481,0.031015,8.128860e-02,5.689471e-01,2.022859e-01,3.537025e-17,1.304954e-02
2017-06-30,0.016359,0.010986,0.020599,0.023493,0.024792,7.467635e-02,6.126554e-01,1.797438e-01,2.732358e-02,9.371944e-03
2017-09-29,0.012824,0.008556,0.013929,0.018715,0.014781,9.406223e-02,6.421553e-01,1.949766e-01,1.975925e-21,3.262872e-17
2017-12-29,0.012770,0.007955,0.015515,0.017659,0.014728,1.330338e-01,4.828685e-01,2.723119e-01,3.429983e-02,8.859851e-03
2018-03-30,0.011585,0.008083,0.009505,0.010614,0.013679,1.807059e-01,4.589145e-01,2.583381e-01,3.957498e-02,9.000161e-03
2018-06-29,0.011730,0.007949,0.010843,0.011477,0.014791,1.449649e-01,5.236151e-01,2.210527e-01,4.426204e-02,9.314841e-03


In [30]:
# 查看回测统计
rp_result.stats()

Start                                  2015-09-01 00:00:00
End                                    2026-03-06 00:00:00
Period                                  2550 days 00:00:00
Start Value                                      1000000.0
End Value                                    2262218.33624
Total Return [%]                                126.221834
Benchmark Return [%]                            144.795482
Max Gross Exposure [%]                               100.0
Total Fees Paid                                        0.0
Max Drawdown [%]                                  7.917383
Max Drawdown Duration                    479 days 00:00:00
Total Trades                                          9739
Total Closed Trades                                   9729
Total Open Trades                                       10
Open Trade PnL                                26481.055755
Win Rate [%]                                     81.426663
Best Trade [%]                                   77.5420

In [31]:
# 可视化结果
rp_viz = BacktestVisualizer(rp_result)
rp_viz.print_metrics()
rp_viz.plot_summary()


Risk Parity 策略表现
总收益率: 126.22%
年化收益率: 12.40%
年化波动率: 6.07%
夏普比率: 1.956
索提诺比率: 3.177
Calmar比率: 1.566
Omega比率: 1.596
最大回撤: -7.92%



In [27]:
# 权重热力图
rp_viz.plot_weights_heatmap(freq='QE')

In [32]:
# 权重变化分析与调仓点标注
# 分析模型的重要调仓时机
rp_viz.plot_weight_changes_analysis(threshold=0.03)

In [33]:
# 资产价格走势与权重变化组合分析
# 这个图可以帮助你看到模型何时对各个资产进行加减仓
rp_viz.plot_assets_and_weights(
    price_df,           # 原始价格数据
    freq='W',           # 按周显示权重变化
    top_n=8             # 显示权重最大的8个资产
)

### 如何解读这些图表

**资产走势与权重分析图**：
- **上半部分**：各资产价格走势（归一化到100），帮助理解资产的历史表现
- **下半部分**：对应的权重配置变化，显示模型何时加减仓
- **分析要点**：
  - 当某资产价格下跌时，模型是否增加权重（抄底）或减少权重（止损）
  - 当某资产价格上涨时，模型是否减少权重（获利了结）或增加权重（追涨）
  - 权重变化频率反映策略的调仓灵敏度

**权重变化分析图**：
- 显示所有资产的权重配置时间序列
- 标注重要的调仓点（权重变化超过阈值）
- 统计月度调仓频率，帮助理解策略的活跃度

## 3. 均值方差策略回测

In [15]:
# 创建均值方差策略（最大化夏普比率）
mv_strategy = MeanVarianceStrategy(
    lookback=60,
    rebalance_freq='ME'
)

# 运行回测
mv_result = engine.run(mv_strategy, price_df)

# 可视化
mv_viz = BacktestVisualizer(mv_result)
mv_viz.print_metrics()
mv_viz.plot_summary()

d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`




Mean Variance 策略表现
总收益率: 111.10%
年化收益率: 11.29%
年化波动率: 10.39%
夏普比率: 1.082
索提诺比率: 1.510
Calmar比率: 0.696
Omega比率: 1.182
最大回撤: -16.22%



## 4. 策略对比

In [16]:
# 创建多个策略进行对比
strategies = [
    RiskParityStrategy(lookback=60, rebalance_freq='ME'),
    RiskParityStrategy(lookback=120, rebalance_freq='QE'),
    MeanVarianceStrategy(lookback=60, rebalance_freq='ME'),
]

names = ['风险平价(60日/月)', '风险平价(120日/季)', '均值方差(60日/月)']

# 运行所有策略
results = []
for strategy, name in zip(strategies, names):
    result = engine.run(strategy, price_df)
    results.append(result)
    print(f"{name}: 总收益={result.metrics['total_return']*100:.2f}%, 夏普={result.metrics['sharpe_ratio']:.3f}")

d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



风险平价(60日/月): 总收益=68.98%, 夏普=1.124


d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



风险平价(120日/季): 总收益=139.70%, 夏普=2.157


d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



均值方差(60日/月): 总收益=111.10%, 夏普=1.082


In [17]:
# 累计收益对比图
BacktestVisualizer.compare_results(results, names=names)

In [ ]:
# 指标对比表
comparison_table = BacktestVisualizer.compare_metrics_table(results, names)
comparison_table

""


## 5. 创建自定义策略

继承 `BaseStrategy` 即可创建自己的策略

In [ ]:
from portfolio_backtest.strategies.base import BaseStrategy

class EqualWeightStrategy(BaseStrategy):
    """等权重策略 - 自定义策略示例"""
    
    def __init__(self, rebalance_freq='ME'):
        super().__init__(name="Equal Weight", rebalance_freq=rebalance_freq)
        self.rebalance_freq = rebalance_freq
    
    def generate_weights(self, price_df, rebalance_mask=None):
        price_df = self.validate_data(price_df)
        
        if rebalance_mask is None:
            rebalance_dates = self.get_rebalance_dates(price_df, self.rebalance_freq)
            rebalance_mask = pd.Series(
                price_df.index.isin(rebalance_dates),
                index=price_df.index
            )
        
        n_assets = price_df.shape[1]
        equal_weight = 1.0 / n_assets
        
        rebalance_dates = price_df.index[rebalance_mask]
        weights_list = [np.full(n_assets, equal_weight) for _ in rebalance_dates]
        
        return pd.DataFrame(
            weights_list,
            index=rebalance_dates,
            columns=price_df.columns
        )

# 使用自定义策略
ew_strategy = EqualWeightStrategy(rebalance_freq='ME')
ew_result = engine.run(ew_strategy, price_df)

ew_viz = BacktestVisualizer(ew_result)
ew_viz.print_metrics()


Equal Weight 策略表现
总收益率: 145.00%
年化收益率: 13.69%
年化波动率: 10.58%
夏普比率: 1.265
索提诺比率: 1.794
Calmar比率: 0.844
Omega比率: 1.209
最大回撤: -16.22%



d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

